# 09 - OOD USDC Depeg Test (Placeholder)

**Purpose.** Documents the planned out-of-distribution stress test on the **March 2023 USDC depeg week** (out of the project's primary Nov 2024 - Apr 2026 window). The expectation is that the forecaster degrades on that window and that the MCDM allocator should fall back gracefully via the f_Stab(dTVL) and f_Risk(u) factors.

**Data source (planned).** *Rules to Rewards* 2025 dataset (Aave historical states, public). See PROJECT_2_PLAN.md S10 Ablation 14.

**Prerequisites.** Once the dataset is downloaded, place it under `data/cached/ood/`. Until then this notebook prints stubs.

**Expected runtime.** N/A until the dataset is fetched.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


## 1. Try to load the OOD slice

In [ ]:
from pathlib import Path
import pandas as pd

OOD_PATH = ROOT / 'data' / 'cached' / 'ood' / 'usdc_depeg_mar2023.parquet'
try:
    df_ood = pd.read_parquet(OOD_PATH)
    print(f'[real] loaded {len(df_ood):,} rows from {OOD_PATH}')
    HAS_OOD = True
except FileNotFoundError:
    print(f'[stub] {OOD_PATH} not present.')
    print('Fetch instructions: see docs/CREDENTIALS_SETUP.md and the\n'
          '"Rules to Rewards" 2025 dataset on Aave historical states.')
    HAS_OOD = False
    df_ood = None


## 2. Plot the depeg week if data is available

In [ ]:
import matplotlib.pyplot as plt

if HAS_OOD and df_ood is not None:
    DEPEG_START = '2023-03-10'
    DEPEG_END   = '2023-03-17'
    window = df_ood.loc[DEPEG_START:DEPEG_END]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(window.index, window['r_aave'] * 100, label='Aave APY %')
    if 'r_compound' in window.columns:
        ax.plot(window.index, window['r_compound'] * 100, label='Compound APY %')
    ax.set_title('USDC depeg week (Mar 10-17, 2023)')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('No OOD data; skipping the depeg-week plot.')


## 3. Forecast degradation expectation

In [ ]:
# Expected behaviour, from PROJECT_2_PLAN.md S10 Ablation 14:
#
#   1. The DL forecaster's directional accuracy drops below 55%
#      (the minimum acceptable threshold) on the depeg week.
#   2. f_Risk(u) rises as borrowers exit, pushing utilization down
#      and lowering the MCDM risk-factor for both protocols.
#   3. f_Stab(dTVL) penalises both protocols simultaneously because
#      TVL contracts sharply at the depeg shock.
#   4. Net effect: MCDM hysteresis prevents thrashing; the strategy
#      sits on its prior allocation and weathers the week.
#
# Concrete metric to compute once data is in:
#   - hit_rate_ood = mean(sign(r_aave - r_comp) == sign(r_hat_aave - r_hat_comp))
#   - delta_apy_ood vs single-protocol benchmark
print('Forecast-degradation harness defined; awaiting OOD data fetch.')


## 4. Citation block

In [ ]:
CITATION = '''\
Rules to Rewards: a Reinforcement Learning Dataset for DeFi Lending.\
  Hosted on figshare; covers Aave historical states including the\
  March 2023 USDC depeg shock. Used here for ablation 14 (OOD test)\
  per PROJECT_2_PLAN.md S10.\
'''
print(CITATION)


## 5. Code stub: forecaster degradation harness

In [ ]:
def evaluate_ood(forecaster, df_ood, horizon: int = 12) -> dict:
    """Compute directional accuracy + bias on the OOD slice.

    Stub - actual implementation will iterate the rolling window over
    df_ood and accumulate per-protocol residuals.
    """
    return {
        'directional_accuracy': float('nan'),
        'bias_aave':            float('nan'),
        'bias_compound':        float('nan'),
        'rmse_aave':            float('nan'),
        'rmse_compound':        float('nan'),
    }

print(evaluate_ood(None, df_ood))


## Next steps

- Fetch the Rules-to-Rewards 2025 dataset and persist the depeg slice   under `data/cached/ood/usdc_depeg_mar2023.parquet`.
- Wire the trained ONNX forecaster into `evaluate_ood()` and re-run.
- Tabulate degradation vs the in-window test set; expect the f_Risk +   f_Stab MCDM factors to prevent catastrophic allocation behaviour.

Relevant plan section: **PROJECT_2_PLAN.md S10 Ablation 14.**
